## Emergent Abilities

Capabilities that appear suddenly at a certain model scale, but were essentially absent in smaller versions of the same architecture.
Not a gradual improvement — a threshold effect. Below a certain size, the model simply cannot do the task. Above it, the ability appears, sometimes abruptly.

## How it actually works

**What "emergent" actually means here:**
Most properties of a model — accuracy, fluency, general usefulness — improve smoothly
as you increase model size, training data, and compute. You see a steady upward curve.

Emergent abilities do not follow that smooth curve. They sit flat near zero
for a long stretch as the model gets bigger, then rise sharply once a certain scale is crossed —
producing a curve that looks closer to a step function than a smooth line.

**Examples that have been observed:**
- Multi-step arithmetic — small models fail almost completely; past a certain scale, accuracy jumps
- Chain-of-thought reasoning — the ability to benefit from "think step by step" prompting
  only shows up reliably above a certain model scale
- Few-shot learning — learning a new pattern from just 2-3 examples in the prompt,
  without any weight updates at all, only becomes reliable at larger scales

**Why this happens — the honest answer is nobody fully agrees:**

**Theory 1 — threshold of representational capacity:**
Some tasks require the model to represent multiple related concepts simultaneously
and combine them in a specific way. Below a certain number of parameters,
there simply is not enough capacity to represent that combination at all.
Cross a threshold, and suddenly there is just enough room — and the ability appears.

**Theory 2 — compression and abstraction:**
Larger models trained on more data tend to learn more ABSTRACT representations
rather than memorising surface-level patterns. Tasks that require abstraction
(like generalising an arithmetic rule rather than memorising specific sums)
only become solvable once the model has learned that abstraction —
which itself may require a certain scale to develop.

**Theory 3 — measurement artefact:**
Some researchers argue part of this is how we MEASURE performance, not the model itself.
If a task is graded strictly pass/fail (the exact right answer or nothing),
a model that is "almost" capable will score 0% right up until it crosses some internal threshold,
then jump to high accuracy — even if its underlying competence was actually improving gradually
the whole time. A more forgiving, partial-credit metric might show a smoother curve
on the exact same models.

## Where it breaks or gets dangerous

**You cannot reliably predict emergent abilities in advance:**
This is the single most important practical consequence. You cannot look at a small model's
performance on a task and confidently extrapolate what a 10x larger version will be able to do.
The capability might still be flat at 10x the size, or it might appear suddenly.
This is exactly why frontier AI labs keep scaling models — partly to discover capabilities
nobody could have predicted from smaller versions.

**Implication for safety and evaluation:**
If new abilities can appear suddenly and unpredictably at scale, that includes abilities
nobody specifically tested for — potentially including abilities relevant to misuse,
not just helpful new skills. This is part of why responsible scaling and careful evaluation
at each new model size matters — you genuinely do not know everything a bigger model
might suddenly be able to do until you test it directly.

**Common mistake — assuming a smaller, cheaper model can "basically" do what a larger one does:**
If a capability is genuinely emergent, a smaller model is not a slightly-worse version
of the larger one for that specific task — it may be near USELESS for it, even if it performs
reasonably well on other tasks. Choosing model size based only on cost without testing
the specific capability you need can produce dramatically worse results than expected.

**Why this matters when picking models for production (directly relevant to your own pipeline):**
You deliberately use Haiku for the Drafter (cheaper, faster) and Sonnet for the Strategist,
Voice Guardian, and Scorer (more capable). This is the right instinct — but it also means
you should specifically test whether the Drafter's task (drafting a raw insight from D365 context)
holds up at the smaller model's scale, rather than assuming "it is just a smaller version
of the same intelligence." Some tasks genuinely degrade non-linearly when you drop model size,
not just by a predictable, smaller percentage.

In [ ]:
# Emergent Abilities — conceptual illustration of smooth vs threshold-shaped performance curves

import numpy as np

def smooth_capability(model_size):
    """
    Simulates a capability that improves smoothly with scale —
    e.g., general fluency, basic factual recall.
    """
    return 1 / (1 + np.exp(-0.02 * (model_size - 200)))  # smooth sigmoid, gradual improvement

def emergent_capability(model_size, threshold=500, sharpness=0.05):
    """
    Simulates a capability that stays near zero, then jumps sharply
    past a threshold — e.g., multi-step arithmetic, chain-of-thought reasoning.
    """
    return 1 / (1 + np.exp(-sharpness * (model_size - threshold)))

# Simulate a range of model sizes (in arbitrary "scale units")
model_sizes = np.array([50, 100, 150, 200, 300, 400, 500, 600, 700, 800, 1000])

print(f"{'Model Size':<12}{'Smooth Capability':<22}{'Emergent Capability':<22}")
print("-" * 56)
for size in model_sizes:
    smooth = smooth_capability(size)
    emergent = emergent_capability(size)
    print(f"{size:<12}{smooth:<22.3f}{emergent:<22.3f}")

print()
print("Notice: smooth capability rises gradually across the whole range.")
print("Emergent capability stays near 0.0 for most of the small-to-mid sizes,")
print("then rises sharply only in the 400-700 range — a threshold, not a gradual climb.")
print()
print("This is why testing a capability ONLY on small/cheap models can be misleading —")
print("a capability sitting at 0.02 might jump to 0.85 well before you expect it to,")
print("or might genuinely stay near zero until a much larger scale than assumed.")

## My D365 analogy

Emergent abilities are like the difference between adding more clerks to a finance team
versus reaching the headcount and structure needed to actually run cross-entity consolidation.

Adding clerk after clerk smoothly increases how many invoices the team processes per day —
a steady, predictable improvement. That is the smooth capability curve.

But something like running a full multi-entity, multi-currency consolidation close
is not just "more of the same work, done by more people." Below a certain team size,
structure, and tooling, it is simply not possible — not slow, not low-quality, just not happening.
Cross a threshold of capability, headcount, and process maturity, and suddenly
the entire consolidation close becomes achievable, sometimes within a single year's transformation.

You cannot predict exactly when that threshold will be crossed just by extrapolating
from a smaller team's performance on simpler tasks. You find out by actually building
toward it and testing whether the capability has emerged yet.

This is the same uncertainty AI researchers face with model scale —
and it is part of why scaling up is still treated as partly empirical,
not something you can fully predict from theory alone.b

## LinkedIn post idea

A small model fails completely at multi-step arithmetic. Make it bigger, and at some point —
not gradually, but almost suddenly — it starts getting the answers right.

This is called an emergent ability, and it is one of the strangest things about scaling AI models.
Most capabilities improve smoothly as models get bigger. A few do not.
They sit flat near zero, then jump sharply once a certain scale is crossed.

Nobody fully agrees on why. Maybe the model needs enough capacity to represent
a combination of concepts at once. Maybe it is about how the task gets measured,
not just the model itself. The honest answer is: this is still being actively studied.

What it means practically is unsettling in a useful way: you cannot always predict
what a bigger model will suddenly be capable of, just by looking at a smaller one.
You have to actually test it.

I think about this every time I choose between a cheaper, faster model and a more capable one
for a specific task in a pipeline. "Slightly worse" is not always the right assumption.
Sometimes the smaller model is not slightly worse at a specific task — it simply cannot do it yet.